🇧🇷 Notebook: Mascaramento Bidirecional com BERTimbau
Este script utiliza a pipeline de fill-mask da Hugging Face para realizar o Masked Language Modeling (MLM).

In [30]:
# Célula 1: Instalação e Importação
import torch
import pandas as pd
import re
from datasets import Dataset, load_dataset
from evaluate import load as load_metric 
from transformers import (
    AutoModelForSequenceClassification, # Modelo para Classificação (BERT)
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from peft import LoraConfig, get_peft_model, TaskType

# 1. Variáveis Globais
MODEL_NAME = 'neuralmind/bert-base-portuguese-cased' 
BOOK_PATH = './biblia.txt' # Altere para o caminho do seu arquivo
OUTPUT_DIR = "./results_bertimbau_ft"

In [2]:
from huggingface_hub import login
# Executa o login interativo:
login()

📖 Etapa 1: Preparação do Dataset e Tokenização
1. Carregamento, Limpeza, **Normalização (Correção)** do Texto
Esta etapa transforma o texto bruto em um dataset formatado com rótulos (labels).

In [31]:
# Célula 2: Carregamento, Limpeza e Combinação de Dados (Arquivos Locais - CORRIGIDO)

import re
from datasets import Dataset, concatenate_datasets, ClassLabel, Features, Value

# Assumindo que BOOK_PATH está definido, definimos o caminho para o novo arquivo.
NON_BOOK_PATH = './nao_biblico.txt'


# Mantenha a função de normalização corrigida:
def normalize_text_for_bert(text):
    """Aplica limpeza estrutural e normalização ortográfica (Português arcaico -> moderno)."""
    text = re.sub(r'[\d\s]+:\d+\s*', ' ', text, flags=re.MULTILINE)
    text = re.sub(r'\s{2,}', ' ', text).strip() 
    
    # Remoção de Itálico e Correções Ortográficas
    text = text.replace('_', '')
    text = text.replace('Christo', 'Cristo').replace('peccado', 'pecado')
    text = text.replace('pae', 'pai').replace('Pae', 'Pai').replace('sciencia', 'ciência')
    text = text.replace('circumcisão', 'circuncisão').replace('hypocrisia', 'hipocrisia')
    text = text.replace('phrophetas', 'profetas').replace('espirito', 'espírito')
    text = text.replace('adopção', 'adoção').replace('coherdeiros', 'co-herdeiros')
    text = text.replace('elle', 'ele')
    
    text = re.sub(r'(_\S+?_)', lambda m: m.group(1).replace('_', ''), text)
    
    return text

# Função auxiliar para carregar e processar textos do arquivo local
def load_and_process_file(file_path, label_id):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            raw_text = f.read()
    except FileNotFoundError:
        raise FileNotFoundError(f"❌ ERRO CRÍTICO: Arquivo '{file_path}' não encontrado. Por favor, crie-o/verifique o caminho.")
    
    normalized_text = normalize_text_for_bert(raw_text)
    
    # Salva o texto normalizado (Conforme solicitado)
    if label_id == 1: 
        output_normalized_path = './biblia_normalizada.txt'
        with open(output_normalized_path, 'w', encoding='utf-8') as f:
            f.write(normalized_text)
        print(f"✅ Texto bíblico normalizado salvo em '{output_normalized_path}'")
    
    # Divide por pontuação terminal e relaxa o filtro para capturar mais amostras
    single_line_text = normalized_text.replace('\n', ' ').strip()
    text_chunks = re.split(r'(?<=[.!?])\s+', single_line_text)

    # Filtra amostras muito curtas (agora com um limite menor para evitar o erro)
    processed_chunks = [
        chunk.strip()
        for chunk in text_chunks
        if len(chunk.strip()) > 20 # Limite de 20 caracteres (antes 50)
    ]
    
    return Dataset.from_dict({'text': processed_chunks, 'label': [label_id] * len(processed_chunks)})


# --- 1. Carregar Classe 1 (Bíblico) ---
biblical_dataset = load_and_process_file(BOOK_PATH, 1)

# --- 2. Carregar Classe 0 (Não Bíblico) de arquivo local ---
non_biblical_dataset = load_and_process_file(NON_BOOK_PATH, 0)


# --- 3. Balancear e Combinar ---
min_samples = min(len(biblical_dataset), len(non_biblical_dataset))

if min_samples < 50: # Aumenta o limite mínimo para garantir um bom treino
    raise ValueError(f"❌ Dados insuficientes: Apenas {min_samples} amostras utilizáveis por classe. Adicione mais texto aos arquivos.")

print(f"\nAmbos os datasets serão truncados para {min_samples} amostras (Balanceamento).")

# Trunca ambos os datasets e combina
biblical_dataset = biblical_dataset.select(range(min_samples))
non_biblical_dataset = non_biblical_dataset.select(range(min_samples))

dataset = concatenate_datasets([biblical_dataset, non_biblical_dataset])
dataset = dataset.shuffle(seed=42) 


# 4. Converter a Coluna 'label' para ClassLabel para permitir Estratificação
print("Convertendo a coluna 'label' para ClassLabel...")
new_features = Features({
    'text': Value('string'),
    'label': ClassLabel(num_classes=2, names=['Não Bíblico', 'Bíblico'])
})
dataset = dataset.cast(new_features)


# 5. Dividir para Treino e Teste
train_test_split = dataset.train_test_split(test_size=0.1, seed=42, stratify_by_column="label") 
train_dataset = train_test_split['train']
eval_dataset = train_test_split['test']

print(f"\nTotal de amostras: {len(dataset)}")
print(f"Tamanho do treino: {len(train_dataset)}")
print(f"Amostras Bíblicas (Label 1) no Treino: {sum(train_dataset['label'])} / Não Bíblicas (Label 0): {len(train_dataset) - sum(train_dataset['label'])}")

✅ Texto bíblico normalizado salvo em './biblia_normalizada.txt'

Ambos os datasets serão truncados para 218 amostras (Balanceamento).
Convertendo a coluna 'label' para ClassLabel...


Casting the dataset:   0%|          | 0/436 [00:00<?, ? examples/s]


Total de amostras: 436
Tamanho do treino: 392
Amostras Bíblicas (Label 1) no Treino: 196 / Não Bíblicas (Label 0): 196


📖 Célula 2: Carregamento, Limpeza e Normalização dos Dados (Corrigida)
Esta célula define a função de limpeza e normalização ortográfica e, em seguida, carrega, processa e divide os dados. A lógica de divisão foi ajustada para gerar mais amostras

In [32]:
# Célula 3: Tokenização e Preparação Final

# Carregar Tokenizador
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# O BERT não usa EOS/PAD, mas vamos garantir o PAD token para o Data Collator
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})

def tokenize_function(examples):
    # Truncation=True garante que o texto não exceda o max_length do BERT (512)
    return tokenizer(examples["text"], truncation=True, max_length=256)

# Mapear a função de tokenização para os datasets
tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_eval_dataset = eval_dataset.map(tokenize_function, batched=True)

# Remover a coluna 'text' original para deixar apenas os IDs de token
tokenized_train_dataset = tokenized_train_dataset.remove_columns(["text"])
tokenized_eval_dataset = tokenized_eval_dataset.remove_columns(["text"])

Map:   0%|          | 0/392 [00:00<?, ? examples/s]

Map:   0%|          | 0/44 [00:00<?, ? examples/s]

In [33]:
# Célula 4: Configuração do Modelo e LoRA

# 1. Carregar Modelo (2 labels para teste binário: 0/1)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=2,
    use_safetensors=True,
    from_tf=False,
)
# 2. Configuração LoRA para Classificação de Sequência
lora_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=32,
    bias="none",
    task_type=TaskType.SEQ_CLS, # TaskType específico para Classificação
)

# 3. Aplicar LoRA ao modelo
model = get_peft_model(model, lora_config)
print("\n--- BERTimbau configurado com LoRA ---")
model.print_trainable_parameters()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



--- BERTimbau configurado com LoRA ---
trainable params: 1,181,186 || all params: 110,105,860 || trainable%: 1.0728


In [36]:
# Célula 5: Configuração e Início do Treinamento

# 1. Data Collator (Padding Dinâmico)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 2. Argumentos de Treinamento (Otimizados para convergência rápida)
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    optim="adamw_torch",
    logging_steps=10,
    learning_rate=5e-5,
    num_train_epochs=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=True, # Usado para BERT
    dataloader_drop_last=True,
    load_best_model_at_end=True,
)

# 3. Inicializar Trainer (Usando o Trainer padrão)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# 4. Iniciar Treinamento
print("\n--- Iniciando Treinamento com BERTimbau ---")
trainer.train()

# 5. Salvar o Modelo Ajustado (apenas os pesos LoRA)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"\n✅ Fine-Tuning concluído! Modelo e Tokenizador salvos em {OUTPUT_DIR}")

/tmp/ipykernel_2332/2584248159.py:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(



--- Iniciando Treinamento com BERTimbau ---


Epoch,Training Loss,Validation Loss
1,0.473400,0.388364
2,0.356400,0.295624
3,0.288500,0.236931
4,0.239500,0.205457
5,0.239200,0.195799



✅ Fine-Tuning concluído! Modelo e Tokenizador salvos em ./results_bertimbau_ft


In [37]:
# Célula 6: Inferência e Teste de Domínio

from peft import PeftModel
import torch.nn.functional as F

# 1. Carregar modelo base e pesos LoRA
BASE_MODEL_NAME = 'neuralmind/bert-base-portuguese-cased'

base_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL_NAME, 
    num_labels=2,
    use_safetensors=True,
    from_tf=False, 
)
model_to_infer = PeftModel.from_pretrained(base_model, OUTPUT_DIR).eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_to_infer.to(device)


def classificar_texto(texto):
    inputs = tokenizer(texto, return_tensors="pt", truncation=True, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model_to_infer(**inputs)
    
    probabilities = F.softmax(outputs.logits, dim=1)[0]
    
    print(f"\nTEXTO: {texto}")
    print(f"Prob. Classe 0 (Outro Domínio): {probabilities[0].item():.4f}")
    print(f"Prob. Classe 1 (Domínio Bíblico): {probabilities[1].item():.4f}")

# 2. Testes

# Texto do domínio (Bíblia)
texto_biblico = "Disse-lhe Jesus: Eu sou o caminho, e a verdade, e a vida."
classificar_texto(texto_biblico)

# Texto fora do domínio
texto_secular = "A nova lei tributária será votada na próxima terça-feira pelo congresso."
classificar_texto(texto_secular)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.



TEXTO: Disse-lhe Jesus: Eu sou o caminho, e a verdade, e a vida.
Prob. Classe 0 (Outro Domínio): 0.2995
Prob. Classe 1 (Domínio Bíblico): 0.7005

TEXTO: A nova lei tributária será votada na próxima terça-feira pelo congresso.
Prob. Classe 0 (Outro Domínio): 0.7951
Prob. Classe 1 (Domínio Bíblico): 0.2049


In [38]:
# Célula de Inferência com Máscara - CORRIGIDA
from transformers import BertForMaskedLM, BertTokenizer
from peft import PeftModel # <--- Importação necessária
import torch
import torch.nn.functional as F

# Usamos o modelo Base e o Diretório de Saída
BASE_MODEL_NAME = "neuralmind/bert-base-portuguese-cased" 
OUTPUT_DIR = "./results_bertimbau_ft" # Deve ser o mesmo de 'Célula 1'
LABELS = {0: "Não Bíblico", 1: "Bíblico"} # <--- VARIÁVEL LABELS DEFINIDA AQUI
import torch
print(f"Versão atual do PyTorch: {torch.__version__}")
try:
    base_model = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL_NAME, 
        num_labels=2, 
        use_safetensors=True, 
        ignore_mismatched_sizes=True
    )
    
    # 2. Carregar os pesos LoRA (adaptadores) e fundi-los ao modelo base
    # Isso transforma o 'base_model' no 'modelo_treinado'
    # Nota: Usamos o .eval() e .merge_and_unload() para preparar o modelo para o pipeline de inferência
    model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
    model = model.merge_and_unload() # Fusão dos pesos LoRA ao base
    model.eval()

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    print(f"✅ Modelo ajustado (LoRA fundido) e Tokenizador carregados em {device}.")
    
    # 3. Função de Previsão de Classe
    def classificar_frase(frase):
        # Tokeniza a frase
        inputs = tokenizer(frase, return_tensors="pt", truncation=True, padding=True).to(device)
        
        # Passa pelo modelo e obtém os logits
        with torch.no_grad():
            outputs = model(**inputs)
        
        # Converte logits para probabilidades (Softmax)
        probabilities = F.softmax(outputs.logits, dim=-1)[0]
        
        # Obtém a classe com maior probabilidade
        predicted_class_id = probabilities.argmax().item()
        
        print("-" * 50)
        print(f"FRASE: '{frase}'")
        
        for i, prob in enumerate(probabilities):
            label = LABELS.get(i, f"Label {i}")
            print(f"  {label}: {prob.item() * 100:.2f}%")
        
        print(f"\n-> PREVISÃO FINAL: {LABELS[predicted_class_id]} ({probabilities.max().item() * 100:.2f}%)")
        print("-" * 50)
        return predicted_class_id, probabilities

    # 4. Testes de Inferência
    
    # Exemplo 1: Domínio Bíblico (Esperado: 1 - Bíblico)
    frase_1 = "E o Senhor disse a Moisés: 'Eu farei chover pão dos céus para vós.'"
    classificar_frase(frase_1)
    
    # Exemplo 2: Domínio Não-Bíblico (Esperado: 0 - Não Bíblico)
    frase_2 = "Ele comprou um carro novo na loja da esquina."
    classificar_frase(frase_2)

    # Exemplo 3: Frase com vocabulário antigo/formal (Teste de generalização)
    frase_3 = "Porque a inclinação da carne é morte, mas a inclinação do espírito é vida e paz."
    classificar_frase(frase_3)
    
    # Exemplo 4: Frase neutra ou moderna (Teste de distinção)
    frase_4 = "O estudo da engenharia de software é essencial para a indústria."
    classificar_frase(frase_4)
except Exception as e:
    print(f"\n❌ Erro Crítico após a atualização: {e}")
    print("Verifique se o seu ambiente foi reiniciado após o 'pip install --upgrade torch'.")
    

Versão atual do PyTorch: 2.6.0+cu124


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


✅ Modelo ajustado (LoRA fundido) e Tokenizador carregados em cuda.
--------------------------------------------------
FRASE: 'E o Senhor disse a Moisés: 'Eu farei chover pão dos céus para vós.''
  Não Bíblico: 28.57%
  Bíblico: 71.43%

-> PREVISÃO FINAL: Bíblico (71.43%)
--------------------------------------------------
--------------------------------------------------
FRASE: 'Ele comprou um carro novo na loja da esquina.'
  Não Bíblico: 68.93%
  Bíblico: 31.07%

-> PREVISÃO FINAL: Não Bíblico (68.93%)
--------------------------------------------------
--------------------------------------------------
FRASE: 'Porque a inclinação da carne é morte, mas a inclinação do espírito é vida e paz.'
  Não Bíblico: 29.22%
  Bíblico: 70.78%

-> PREVISÃO FINAL: Bíblico (70.78%)
--------------------------------------------------
--------------------------------------------------
FRASE: 'O estudo da engenharia de software é essencial para a indústria.'
  Não Bíblico: 82.64%
  Bíblico: 17.36%

-> P

In [40]:
# Célula de Inferência de Máscara (fill-mask)

from transformers import BertForMaskedLM, AutoTokenizer, pipeline
from peft import PeftModel
import torch

# 1. Variáveis
OUTPUT_DIR = "./results_bertimbau_ft" # Diretório onde os adaptadores LoRA foram salvos
BASE_MODEL_NAME = "neuralmind/bert-base-portuguese-cased"

# 2. Carregar o Modelo Base como BertForMaskedLM
try:
    # Carrega o modelo base com a 'cabeça' de Masked Language Model (MLM)
    base_model = BertForMaskedLM.from_pretrained(
        BASE_MODEL_NAME, 
        use_safetensors=True, 
        ignore_mismatched_sizes=True # Permite ignorar a 'cabeça' de classificação antiga
    )
    
    # 3. Carrega os adaptadores LoRA salvos do seu fine-tuning de SEQ_CLS
    # e aplica (funde) eles ao BertForMaskedLM base.
    model_peft = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
    model = model_peft.merge_and_unload()
    model.eval()
    
    # Carrega o Tokenizador
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    
    print(f"✅ Modelo ajustado (LoRA fundido) e Tokenizador carregados em {device}.")
    
    # 4. Criar a pipeline de fill-mask
    unmasker = pipeline(
        "fill-mask",
        model=model,
        tokenizer=tokenizer,
        device=0 if torch.cuda.is_available() else -1
    )
    # 5. Função de Previsão de Máscara
    def prever_palavra_mascarada(frase_mascarada, top_k=5):
        # A pipeline gerencia a substituição de [MASK] pelo token correto
        resultados = unmasker(frase_mascarada, top_k=top_k)
        
        print("\n" + "=" * 50)
        print(f"FRASE COM MÁSCARA: {frase_mascarada}")
        print("=" * 50)
        
        for i, res in enumerate(resultados):
            score = res['score'] * 100
            frase_preenchida = res['sequence']
            
            print(f"  {i+1}. {frase_preenchida} (Probabilidade: {score:.2f}%)")

    # 6. Testes
    
    frase_biblica = "O senhor é meu [MASK], e nada me faltará."
    prever_palavra_mascarada(frase_biblica)
    
    frase_jfa = "A Graça e a [MASK] de nosso Senhor Jesus Christo seja convosco."
    prever_palavra_mascarada(frase_jfa)
    
    frase_neutra = "O [MASK] do Brasil é a capital do país."
    prever_palavra_mascarada(frase_neutra)
    
except Exception as e:
    print(f"\n❌ Erro ao carregar o modelo ajustado: {e}")
    print("Verifique se o diretório OUTPUT_DIR e o modelo base estão corretos.")

Some weights of the model checkpoint at neuralmind/bert-base-portuguese-cased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0


✅ Modelo ajustado (LoRA fundido) e Tokenizador carregados em cuda.

FRASE COM MÁSCARA: O senhor é meu [MASK], e nada me faltará.
  1. O senhor é meu amigo, e nada me faltará. (Probabilidade: 23.49%)
  2. O senhor é meu Deus, e nada me faltará. (Probabilidade: 8.53%)
  3. O senhor é meu pai, e nada me faltará. (Probabilidade: 7.38%)
  4. O senhor é meu irmão, e nada me faltará. (Probabilidade: 5.13%)
  5. O senhor é meu Senhor, e nada me faltará. (Probabilidade: 4.91%)

FRASE COM MÁSCARA: A Graça e a [MASK] de nosso Senhor Jesus Christo seja convosco.
  1. A Graça e a Paz de nosso Senhor Jesus Christo seja convosco. (Probabilidade: 72.68%)
  2. A Graça e a Misericórdia de nosso Senhor Jesus Christo seja convosco. (Probabilidade: 13.85%)
  3. A Graça e a paz de nosso Senhor Jesus Christo seja convosco. (Probabilidade: 10.88%)
  4. A Graça e a Glória de nosso Senhor Jesus Christo seja convosco. (Probabilidade: 0.45%)
  5. A Graça e a Vida de nosso Senhor Jesus Christo seja convosco. (Prob